In [1]:
from tool3 import *
device = auto_device()
set_seed(42)

In [4]:
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration
name = 'gogamza/kobart-summarization'
tok = PreTrainedTokenizerFast.from_pretrained(name)
model = BartForConditionalGeneration.from_pretrained(name)
model.eval()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(30000, 768, padding_idx=3)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(30000, 768, padding_idx=3)
      (embed_positions): BartLearnedPositionalEmbedding(1028, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [6]:
doc = ('정부는 오늘 발표한 보고서에서 인공지능 산업이 향후 10년간 연평균 30% 성장할 것으로 전망했다. '
       '특히 한국어 거대 언어모델 개발에 대한 민간 투자가 급증하면서 관련 인력 수요도 빠르게 늘어나고 있다. '
       '전문가들은 데이터 보호와 윤리 가이드라인이 함께 마련되어야 한다고 강조했다.'
       '바람의 나라')

In [7]:
x = tok(doc, return_tensors='pt', max_length=512, truncation=True)
import torch
with torch.no_grad():
    y = model.generate(**x, num_beams=4, max_length=64,
                      length_penalty=1.0)
tok.decode(y[0], skip_speical_tokens=True)

'<s> 정부는 오늘 발표한 보고서에서 인공지능 산업이 향후 10년간 연평균 30% 성장할 것으로 전망했다.</s>'

In [9]:
with torch.no_grad():
    # length_penalty = 1.0, 0.5, 2.0
    y_g = model.generate(**x, do_sample=False, length_penalty=1.0, max_length=64)
    y_b = model.generate(**x, num_beams=3, max_length=64)
    y_s = model.generate(**x, do_sample=True, top_p=0.9, max_length=64)
# hps = [('greedy', y_g), ('beam=4', y_b), ('top-p=0.9', y_s)]
# for tag, y in hps:
#     print(tag, '|', tok.decode(y[0], skip_special_tokens=True))
ref = '인공지능 산업 성장 전망과 한국어 LLM 투자 증가, 윤리 가이드라인 필요성'
pred = tok.decode(y_b[0], skip_special_tokens=True)
round(rouge_l(ref, pred), 3)

0.083